# 一次 SQL 批量检索多个 IVF 向量

DataFusion 和 LakeSoul pgwire 使用相同的原生批量表函数，共享快照、cluster 加载和 FastScan 数据块。此 Notebook 创建独立的本地演示表，运行一次批量 SQL，验证结果后清理。需要编译了原生批 SQL 功能的 LakeSoul Python 扩展，以及可用的 PostgreSQL 元数据库。

完整 API、并发预算与分布式语义见 [原生批量 SQL](../native-batch-sql.md)。

In [1]:
import os
import json
import uuid
import tempfile
from pathlib import Path
import numpy as np
import pyarrow as pa
from lakesoul import LakeSoulCatalog

catalog = LakeSoulCatalog.from_env(namespace="default")
session = catalog.session()
run_id = "native_batch_notebook_" + uuid.uuid4().hex
root = Path(tempfile.mkdtemp(prefix=run_id + "_"))
(root / ".owner").write_text(run_id)
target_name = run_id + "_items"
query_name = run_id + "_queries"
print("演示表:", target_name, query_name)
print("演示目录:", root)


演示表: native_batch_notebook_b9493f637bc04769b2d36e33ff1b626a_items native_batch_notebook_b9493f637bc04769b2d36e33ff1b626a_queries
演示目录: /tmp/native_batch_notebook_b9493f637bc04769b2d36e33ff1b626a_ag6b6blh


## 创建演示数据与索引

示例为 128 个 8 维向量、4 个 IVF cluster、7-bit RaBitQ，目标主键为 Int64 `id`。查询表含唯一 Int64 `qid` 与 Float32 列表 `qvec`。

In [2]:
dim = 8
vectors = np.random.default_rng(7).normal(size=(128, dim)).astype(np.float32)
queries = vectors[[7, 31, 99]] + np.float32(0.001)
query_ids = [101, 205, 999]
vector_type = pa.list_(pa.field("item", pa.float32()), dim)

target = catalog.create_table(
    target_name, path=(root / "items").as_uri(),
    schema=pa.schema([pa.field("id", pa.int64(), False),
                      pa.field("vec", vector_type, False)]),
    primary_keys=["id"], hash_bucket_num=4,
    vector_index=[{"column": "vec", "dim": dim, "nlist": 4,
                   "total_bits": 7, "metric": "L2"}],
)
target.write_arrow(pa.table({
    "id": pa.array(range(len(vectors)), type=pa.int64()),
    "vec": pa.FixedSizeListArray.from_arrays(pa.array(vectors.reshape(-1)), dim),
}))
query_table = catalog.create_table(
    query_name, path=(root / "queries").as_uri(),
    schema=pa.schema([pa.field("qid", pa.int64(), False),
                      pa.field("qvec", vector_type, False)]),
    primary_keys=["qid"], hash_bucket_num=1,
)
query_table.write_arrow(pa.table({
    "qid": pa.array(query_ids, type=pa.int64()),
    "qvec": pa.FixedSizeListArray.from_arrays(pa.array(queries.reshape(-1)), dim),
}))
print("已写入", len(vectors), "个目标向量和", len(queries), "个查询向量")


已写入 128 个目标向量和 3 个查询向量


## 一次 SQL 查询整张 query 表

结果按 `query_id, rank` 排序；每个 query 维护自己的 top-k。`exact` 对 ANN 候选重排，不代表全库精确检索保证。本例探测全部 4 个 cluster，并使用足够的 overfetch，便于与小数据集暴力搜索比较。

In [3]:
sql = f"""
SELECT query_id, rank, id, distance
FROM lakesoul_vector_search_batch_from_table(
  'lakesoul.default.{target_name}', 'vec',
  'lakesoul.default.{query_name}', 'qid', 'qvec',
  5, 4, 'L2', 'exact', 10
)
ORDER BY query_id, rank
"""
print(sql)
results = session.sql(sql)
assert results.num_rows == len(queries) * 5
results



SELECT query_id, rank, id, distance
FROM lakesoul_vector_search_batch_from_table(
  'lakesoul.default.native_batch_notebook_b9493f637bc04769b2d36e33ff1b626a_items', 'vec',
  'lakesoul.default.native_batch_notebook_b9493f637bc04769b2d36e33ff1b626a_queries', 'qid', 'qvec',
  5, 4, 'L2', 'exact', 10
)
ORDER BY query_id, rank

pyarrow.Table
query_id: int64 not null
rank: int32 not null
id: int64 not null
distance: double not null
----
query_id: [[101,101,101,101,101,...,999,999,999,999,999]]
rank: [[1,2,3,4,5,...,1,2,3,4,5]]
id: [[7,40,29,48,124,...,99,58,104,21,114]]
distance: [[0.0028284486652743017,1.3317854758024468,1.356692814063398,1.7025625717607262,1.8802304112484578,...,0.0028284592764864576,1.8587699995779943,2.005003511611261,2.4650190142967916,2.510337532875048]]


In [4]:
rows = results.to_pylist()
for qid, query in zip(query_ids, queries):
    expected = np.argsort(np.sum((vectors - query) ** 2, axis=1), kind="stable")[:5]
    actual = [row for row in rows if row["query_id"] == qid]
    assert [row["rank"] for row in actual] == list(range(1, 6))
    assert [row["id"] for row in actual] == expected.tolist()
    expected_distances = np.sqrt(np.sum((vectors[expected] - query) ** 2, axis=1))
    np.testing.assert_allclose([row["distance"] for row in actual], expected_distances, rtol=1e-5, atol=1e-5)
print("每个query的ID、rank和L2距离均与小数据集基线一致")


每个query的ID、rank和L2距离均与小数据集基线一致


## 少量 query 使用 inline JSON

inline 的 query_id 从0开始。大请求推荐查询表入口，避免构造很长的 SQL 文本。

In [5]:
payload = json.dumps(queries.tolist(), separators=(",", ":"))
inline_sql = f"""
SELECT query_id, rank, id, distance
FROM lakesoul_vector_search_batch(
  'lakesoul.default.{target_name}', 'vec', '{payload}',
  5, 4, 'L2', 'exact', 10
) ORDER BY query_id, rank
"""
inline = session.sql(inline_sql)
assert inline.column("id").to_pylist() == results.column("id").to_pylist()
inline


pyarrow.Table
query_id: int64 not null
rank: int32 not null
id: int64 not null
distance: double not null
----
query_id: [[0,0,0,0,0,...,2,2,2,2,2]]
rank: [[1,2,3,4,5,...,1,2,3,4,5]]
id: [[7,40,29,48,124,...,99,58,104,21,114]]
distance: [[0.0028284486652743017,1.3317854758024468,1.356692814063398,1.7025625717607262,1.8802304112484578,...,0.0028284592764864576,1.8587699995779943,2.005003511611261,2.4650190142967916,2.510337532875048]]


## 配置与执行计划

window 限制每次在内核中保留的 query 数，parallelism 限制请求内 CPU 并行。默认值由受控线程池及工作内存预算决定；设置线程数不会将整个索引复制到各线程。

In [6]:
session.sql("SET lakesoul_vector_batch.query_window = '64'")
session.sql("SET lakesoul_vector_batch.parallelism = '2'")
plan = session.explain(sql)
assert "VectorBatchSearchExec" in plan
print(plan)


+------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| plan_type                                                  | plan                                                                                                                                                                                                                                                                                                                                                                                                                         |
+-------------------------------------------

## 可选：通过 PostgreSQL/pgvector 兼容前端执行同一批查询

设置 `LAKESOUL_PGWIRE_DSN` 后运行。服务须连接同一个元数据库，并能读取上述本地目录（同主机示例）；多节点部署应使用所有 worker 可访问的对象存储。函数返回普通 BIGINT/INTEGER/FLOAT8，不需要注册自定义向量类型。pgwire 的表名写作当前数据库的 `public.*`。

In [7]:
dsn = os.environ.get("LAKESOUL_PGWIRE_DSN")
if dsn:
    import psycopg
    pg_sql = sql.replace(f"lakesoul.default.{target_name}", f"public.{target_name}")
    pg_sql = pg_sql.replace(f"lakesoul.default.{query_name}", f"public.{query_name}")
    with psycopg.connect(dsn, autocommit=True) as connection:
        with connection.cursor() as cursor:
            cursor.execute(pg_sql)
            pg_rows = cursor.fetchall()
    assert len(pg_rows) == results.num_rows
    assert [row[2] for row in pg_rows] == results.column("id").to_pylist()
    print("pgwire一次SQL返回", len(pg_rows), "行，结果一致")
else:
    print("未设置LAKESOUL_PGWIRE_DSN，跳过可选pgwire单元")


未设置LAKESOUL_PGWIRE_DSN，跳过可选pgwire单元


## 清理

仅删除本 Notebook 创建的两张演示表及带有本次运行 owner 标记的临时目录；不触及其它表和共享 cache。若中途运行失败，也应执行此单元。

In [8]:
import shutil
for name in (query_name, target_name):
    assert name.startswith(run_id + "_")
    catalog.drop_table(name, if_exists=True)
assert root.resolve().parent == Path(tempfile.gettempdir()).resolve()
assert (root / ".owner").read_text() == run_id
assert root.name.startswith(run_id + "_")
shutil.rmtree(root.resolve())
print("已清理本次演示资源")


已清理本次演示资源
